# 02.0 — EDA Contract & Experimental Scope

## 1. Mục tiêu
Notebook này **không thực hiện phân tích dữ liệu nặng**.
Nhiệm vụ duy nhất là công bố và xác lập hợp đồng phân tích (**EDA Contract**) chuẩn hóa cho toàn bộ giai đoạn Exploratory Data Analysis:

1. **Canonical Train/Test Split Boundary**:
   - Ranh giới phân chia thời gian bất biến: `2020-05-01` (`src/data/loader.py::time_series_split`).
   - Tổng số ngày dữ liệu sạch: **9.251 ngày** (2001-01-01 đến 2026-04-30), giữ nguyên **36 đặc trưng gốc**.
   - Quyết định Data Quality: Loại bỏ toàn bộ 366 ngày năm 2000 do **Structural / System-driven Missingness** (measurement availability trước 2001) và giữ lại toàn bộ 5 features bị missing trong năm 2000 để EDA đánh giá tính đa cộng tuyến/dư thừa trên dữ liệu đầy đủ.
   - Tập **Train**: từ `2001-01-01` đến `2020-04-30` (**7.060 ngày**).
   - Tập **Test**: từ `2020-05-01` đến `2026-04-30` (**2.191 ngày**) — bị cô lập hoàn toàn (**strictly quarantined**).

2. **Zero Data Leakage Policy**:
   $$\boxed{\text{EDA Input} = \text{train\_df}}$$
   - Tuyệt đối cấm tính toán thống kê (mean, median, skewness, kurtosis), tương quan Spearman, VIF, chu kỳ FFT/MSTL, kiểm định nghiệm đơn vị ADF/KPSS, hoặc ngưỡng cực đoan trên toàn bộ dataset hay tập Test.
   - Bất kỳ thống kê nào học từ dữ liệu phải tuân thủ nghiêm ngặt nguyên tắc `fit(train)` và chỉ áp dụng suy diễn.

3. **Chính sách Ngưỡng mưa (Rainfall Threshold Policy)**:
   - Target variable: `Lượng mưa` (`PRECTOTCORR`, đơn vị mm/ngày).
   - Ngưỡng mưa vết WMO (**WMO Trace Rain Standard**): $\tau = 0.1\text{ mm}$.
   - Khảo sát các ngưỡng ứng viên: $\tau \in \{0.0, 0.1, 0.6, 1.0\text{ mm}\}$.

4. **Tránh Dependency Cycle giữa EDA và Feature Engineering (P0 Resolution)**:
   - Kiến trúc định hướng luồng dữ liệu 1 chiều:
     $$\text{Data Quality} \longrightarrow \text{EDA} \longrightarrow \text{Feature Engineering} \longrightarrow \text{Modeling}$$
   - EDA chỉ nhận `train_df` sau Data Quality (7.060 ngày $\times$ 36 cột).
   - Feature Engineering tiêu thụ `eda_report.json` làm tham số cấu hình độc lập (`wet_season_months`, `vif_filtered_features`, `validated_rain_threshold`), không có chiều phụ thuộc ngược lại.

5. **Danh mục Artifact đầu ra**:
   - `02.1` $\to$ `outputs/threshold_report.json`
   - `02.2` $\to$ `outputs/distribution_report.json`
   - `02.3` $\to$ `outputs/correlation_report.json` (bao gồm đánh giá chuyên biệt 5 biến bức xạ)
   - `02.4` $\to$ `outputs/temporal_structure_report.json`
   - `02.5` $\to$ `outputs/stationarity_report.json`
   - `02.6` $\to$ `outputs/extreme_events_report.json`
   - `02.7` $\to$ `data/processed/eda_report.json`


In [1]:
import sys
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Reconfigure encoding for safety
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8', errors='replace')

# Path resolution
ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.data.loader import DataLoader, time_series_split

print("=" * 70)
print("📜 02.0: ESTABLISHING EDA SCIENTIFIC CONTRACT")
print("=" * 70)

# Load data and verify canonical split
df = DataLoader().load_data()
train_df, test_df = time_series_split(df)

# Assertions for Clean Dataset & Zero Data Leakage
assert len(df) == 9251, f"Expected 9,251 clean rows, got {len(df)}"
assert df.shape[1] == 36, f"Expected 36 columns, got {df.shape[1]}"
assert len(train_df) == 7060, f"Expected 7,060 train rows, got {len(train_df)}"
assert len(test_df) == 2191, f"Expected 2,191 test rows, got {len(test_df)}"
assert train_df['Ngày'].max() < test_df['Ngày'].min(), "Leakage detected: Train and Test overlap!"

print(f"✅ Canonical Split Verified:")
print(f"   • Clean Dataset: {len(df):,} days x {df.shape[1]} columns (2000 excluded via DQ Policy)")
print(f"   • Train: {len(train_df):,} days ({train_df['Ngày'].min().date()} → {train_df['Ngày'].max().date()})")
print(f"   • Test:  {len(test_df):,} days ({test_df['Ngày'].min().date()} → {test_df['Ngày'].max().date()}) [QUARANTINED]")
print(f"   • Target Variable: 'Lượng mưa' (Daily Precipitation in mm)")
print(f"   • Canonical Rain Threshold Policy: tau = 0.1 mm (WMO Trace Rain)")


📜 02.0: ESTABLISHING EDA SCIENTIFIC CONTRACT
✅ Canonical Split Verified:
   • Clean Dataset: 9,251 days x 36 columns (2000 excluded via DQ Policy)
   • Train: 7,060 days (2001-01-01 → 2020-04-30)
   • Test:  2,191 days (2020-05-01 → 2026-04-30) [QUARANTINED]
   • Target Variable: 'Lượng mưa' (Daily Precipitation in mm)
   • Canonical Rain Threshold Policy: tau = 0.1 mm (WMO Trace Rain)


In [2]:
# Export EDA Contract Artifact
OUTPUT_DIR = ROOT_DIR / "notebooks" / "eda" / "outputs"
DATA_OUTPUT_DIR = ROOT_DIR / "data" / "processed" / "eda_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

contract = {
    "canonical_split_date": "2020-05-01",
    "dataset_total_days": int(len(df)),
    "dataset_total_columns": int(df.shape[1]),
    "train_days_count": int(len(train_df)),
    "test_days_count": int(len(test_df)),
    "train_start_date": str(train_df['Ngày'].min().date()),
    "train_end_date": str(train_df['Ngày'].max().date()),
    "test_start_date": str(test_df['Ngày'].min().date()),
    "test_end_date": str(test_df['Ngày'].max().date()),
    "target_col": "Lượng mưa",
    "date_col": "Ngày",
    "rainfall_threshold_policy_mm": 0.1,
    "leakage_rule": "EDA statistics must strictly be computed on train_df only.",
    "pipeline_direction": "Data Quality -> EDA -> Feature Engineering -> Modeling",
    "artifacts_registry": {
        "02.1": "threshold_report.json",
        "02.2": "distribution_report.json",
        "02.3": "correlation_report.json",
        "02.4": "temporal_structure_report.json",
        "02.5": "stationarity_report.json",
        "02.6": "extreme_events_report.json",
        "02.7": "eda_report.json"
    }
}

contract_path = OUTPUT_DIR / "eda_contract.json"
with open(contract_path, "w", encoding="utf-8") as f:
    json.dump(contract, f, indent=2, ensure_ascii=False)

with open(DATA_OUTPUT_DIR / "eda_contract.json", "w", encoding="utf-8") as f:
    json.dump(contract, f, indent=2, ensure_ascii=False)

print(f"💾 EDA Contract exported to: {contract_path}")
print("   • Status: APPROVED & FROZEN")


💾 EDA Contract exported to: D:\DS108_project\notebooks\eda\outputs\eda_contract.json
   • Status: APPROVED & FROZEN
